# Cypher 기본 문법

# langchain-neo4j 를 이용한 연동

- 설치
    ```bash
    pip install langchain-neo4j
    ```

## `Neo4jGraph`

- `from langchain_neo4j import Neo4jGraph`
- `Neo4jGraph`는 Neo4j Python 드라이버를 감싼 래퍼(Wrapper) 클래스로, Neo4j 데이터베이스와 간편하게 상호작용할 수 있는 인터페이스를 제공한다.

- **주요 역할**
  - **Cypher 쿼리 실행** 
    - 읽기/쓰기 Cypher 쿼리를 실행하고 결과를 `list[dict]`로 반환
  - **그래프 스키마 자동 조회** 
    - 노드 레이블·관계 타입·속성 정보를 자동으로 읽어 `schema` 속성에 저장하여 제공
  - **GraphRAG 파이프라인 연계** 
    - `GraphCypherQAChain`, `LLMGraphTransformer` 등과 결합하여 자연어 기반 그래프 질의응답 구성


###  객체 생성 주요 파라미터

- 구문
    ```python
    Neo4jGraph(
        url=None,
        username=None,
        password=None,
        database=None,
        refresh_schema=True,
        enhanced_schema=False,
    )
    ```

    | 파라미터 | 타입 | 기본값 | 설명 |
    |---|---|---|---|
    | `url` | `str \| None` | `None` | Neo4j 서버 URL (환경변수 `NEO4J_URI` 대체 가능) |
    | `username` | `str \| None` | `None` | 인증 사용자명 (환경변수 `NEO4J_USERNAME` 대체 가능) |
    | `password` | `str \| None` | `None` | 인증 패스워드 (환경변수 `NEO4J_PASSWORD` 대체 가능) |
    | `database` | `str \| None` | `"neo4j"` | 접속할 데이터베이스 이름 |
    | `refresh_schema` | `bool` | `True` | `True`이면 객체 생성 시 자동으로 스키마 정보를 조회 |
    | `enhanced_schema` | `bool` | `False` | `True`이면 DB를 스캔하여 속성의 실제 예시값까지 스키마에 포함 |



- **환경변수에 DB 연결정보가 저장된 경우 객체 생성시 설정하지 않아도 자동으로 읽는다.**
    ```python
    # .env 파일에 설정하면 파라미터 생략 가능
    # NEO4J_URI=bolt://localhost:7687
    # NEO4J_USERNAME=neo4j
    # NEO4J_PASSWORD=password
    # NEO4J_DATABASE=db이름

    graph = Neo4jGraph()  # 환경변수에서 자동 읽기
    ```

### 주요 메소드

- **`query()`** — Cypher 쿼리 실행
  - 조회뿐 아니라 쓰기(CREATE/MERGE) 도 실행
  - 구문
      ```python
        graph.query(query: str, params: dict = {}) -> List[Dict]
      ```
  - Cypher 쿼리를 실행하고 결과를 **딕셔너리 리스트**로 반환
  - `params`로 파라미터화 쿼리 지원 (SQL Injection 방지)

      ```python
      # 읽기 쿼리
      result = graph.query(
          "MATCH (m:Movie) WHERE m.genre = $genre RETURN m.title LIMIT 5",
          params={"genre": "Action"}
      )

      # 쓰기 쿼리 (MERGE / CREATE)    
      graph.query("""
          MERGE (m:Movie {title: 'Top Gun'})
          MERGE (a:Actor {name: 'Tom Cruise'})
          MERGE (a)-[:ACTED_IN]->(m)
      """)
      ```

- **`refresh_schema()`** — 스키마 갱신
  - 구문
    ```python
        graph.refresh_schema()
    ```
  - Neo4j DB로부터 노드 레이블, 관계 타입, 속성 정보를 다시 읽어와 갱신
  - 데이터 구조가 바뀐 후 `schema` 정보를 최신 상태로 유지할 때 사용

- **`add_graph_documents()`** — 그래프 문서 삽입
  - `GraphDocument` 객체에 저장한 `Node`와 `Relationship`들을 Database에 저장한다.

    ```python
        graph.add_graph_documents(
            graph_documents: List[GraphDocument]
        )
    ```

    > - **`GraphDocument`**
    >   - 그래프 구조(노드 + 관계)를 담는 컨테이너 객체
    >   - list[Node] 와 list[Relationship] 으로 구성된다. 
    > - **`Node`**
    >   - Node 정보를 담는 dataclass 로 다음 속성을 가진다.
    >     - `id:str|int`: Node 식별자 
    >     - `type:str`: Node의 Label 
    >     - `properties:dict`: Node 속성
    > - **`Relationship`**
    >   - Relationship 정보를 담는 dataclass 로 다음 속성을 가진다.
    >     - `source: Node`: 시작 노드
    >     - `target: Node`: 끝 노드
    >     - `type: str`: Relationship 타입
    >     - `properties: dict`: 관계 속성

- **`close()`** — 연결 종료
    - 구문
        ```python
        graph.close()
        ```
    - Neo4j 드라이버 연결을 명시적으로 닫는다.. 컨텍스트 매니저(`with` 문)를 사용하면 자동으로 호출된다.

###  주요 속성(Instance variable)

| 속성 | 타입 | 설명 |
|---|---|---|
| `schema` | `str` | 현재 DB 스키마 정보 (문자열 형태, LLM 프롬프트에 직접 삽입) |
| `structured_schema` | `dict` | 스키마 정보를 구조화한 딕셔너리 (`node_props`, `rel_props`, `relationships` 키 포함) |

```python
# LLM 프롬프트용 스키마 확인
print(graph.schema)

# 노드 속성 직접 접근
print(graph.structured_schema["node_props"])
```

In [ ]:
import os
from dotenv import load_dotenv
from pprint import pprint

load_dotenv(override=True)

print("NEO4J_URI:", os.getenv("NEO4J_URI"))
print("NEO4J_USERNAME:", os.getenv("NEO4J_USERNAME"))
print("NEO4J_PASSWORD:", os.getenv("NEO4J_PASSWORD"))
print("NEO4J_DATABASE:", os.getenv("NEO4J_DATABASE"))

In [ ]:
from langchain_neo4j import Neo4jGraph
##################################
# 환경변수로 연결설정 정보 저장 후 연결
##################################
graph = Neo4jGraph()

# 연결 확인
print(
    graph.query("RETURN 'hello world' AS result")
)


In [ ]:
# 현재는 생성된 것이 없기 때문에 빈 schema 정보가 출력된다
print(graph.schema)
print("----------------------------------")
pprint(graph.structured_schema)


In [ ]:
############################
# 직접 값을 넣어 연결
############################
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph( 
    url=os.getenv("NEO4J_URI"), 
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE")
)

print(
    graph.query("RETURN 'hello world' AS result")
)

In [ ]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    print(type(constraints)) # list[dict]

    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    print(type(indexes), indexes)
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

# Cypher 쿼리 기초

Cypher는 Neo4j의 그래프 쿼리 언어로, 그래프 패턴을 시각적으로 표현하는 구문을 제공.

## 노드, 속성, 관계 표시
- 노드(Node)는 `()` 소괄호, 관계(Relationship)는 `[]` 대괄호, 속성(Property)는 `{}` 중괄호로 표현
- 라벨은 `:` 콜론으로 시작
- 관계는 반드시 방향(`->` 또는 `<-`)이 있어야 함 (방향 없는 관계는 `CREATE`에서 불가)
- 노드나 관계를 변수에 할당하여 이후에 참조 할 수 있게 한다.
- **예**
   ```cypher
   (n)                     // 모든 노드
   (p:Person)              // Person 레이블의 노드를 조회하고 변수 p에 할당
   (m {name: "Neo4j"})     // name 속성이 "Neo4j"인 노드들을 변수 m에 할당
   (p:Person {age: 30})    // Person 레이블의 age 속성이 30인 노드들을 변수 p에 할당
   -[r:KNOWS]->            // 변수 r에 KNOWS 관계들을 할당
   ```

> ## cypher 주석
>   -  `//`: 한 줄 주석. `//`이 주석의 시작, "엔터"가 주석의 끝.
>   -  `/* */`: 여러 줄 주석. `/*` 이 주석의 시작, `*/`이 주석의 끝.

### **CREATE** 
- 새로운 노드와 관계를 생성한다.
- 동일한 노드가 이미 존재해도 중복 생성되므로 주의해야 한다.
  - 중복 저장을 해결하려면 **constraint**를 설정하거나 **MERGE**를 이용해 저장한다.
- **기본 문법:**

  ```cypher
  // 노드 생성
  CREATE (변수:라벨 {속성키: 속성값, ...})
  ---------------------------------------------------------
  // 관계 생성 (양 노드가 이미 존재할 때)
  MATCH (a), (b)
  WHERE 조건
  CREATE (a)-[:관계타입 {속성}]->(b)
  ---------------------------------------------------------
  // 노드와 관계 한 번에 생성
  CREATE (a:Label1 {...})-[:REL_TYPE]->(b:Label2 {...})
  ```



In [ ]:
#############################################################
#  Neo4j 데이터베이스에 노드 생성하기 (CREATE 구문 사용)
# CREATE 구문은 새로운 노드나 관계를 생성할 때 사용한다.
#############################################################

cypher_query = """
/*
노드 레이블 : Person (사람을 나타내는 레이블)
노드 속성 정의:
  - name: "홍길동" (이름)
  - age: 30 (나이)
  - email: "hong@abc.com" (이메일 주소)
*/
// p는 생성된 노드를 참조하는 변수명.
CREATE (:Person {name: "홍길동", age: 30, email: "hong@abc.com"})
"""


graph.query(cypher_query)

In [ ]:
# CREATE 구문을 사용하여 여러 노드를 한 번에 생성할 수 있다.
cypher_query = """
/* 
노드 레이블 정의:
  - Person: 사람을 나타내는 레이블
  - City: 도시를 나타내는 레이블

Person 노드 속성 정의:
  - name: 사람의 이름 (문자열)
  - age: 사람의 나이 (숫자)
  - gender: 성별 (문자열)
City 노드 속성 정의:
  - name: 도시 이름 (문자열)
  - population: 인구수 (숫자)
*/

CREATE 
  (:Person {name: "김영수", age: 25, gender: "남성"}),
  (:Person {name: "최영희", age: 28, gender: "여성"}),
  (:City {name: "서울", population: 9700000}),
  (:City {name: "부산", population: 3200000}),
  (:City {name: "인천", population: 3050000})
"""

graph.query(cypher_query)

- **관계 표현**: 대시와 대괄호 `-[]->` 사용
   ```cypher
   -->                         // 모든 관계
   -[r:KNOWS]->                // 관계 타입 지정. KNOWS 타입의 관계 
   -[r:LIKES {since: 2020}]->  // 속성을 가진 관계
   <-[r:FOLLOWS]-              // 방향이 반대인 관계
   -[r:FRIENDS|KNOWS]->        // 여러 타입의 관계(OR 조건)
   ```

In [ ]:
#########################################
#  기존 노드 사이에 관계 생성하기
########################################
cypher_query = """
/*
관계 레이블 : 관계의 이름. KNOWS (알고 있다는 관계를 나타냄)
관계 속성 : 관계도 속성을 가질 수 있다. `{name: value, ...}`로 설정.
관계 방향 : a -> b (홍길동이 김철수를 알고 있는 방향성)
관계 속성 값: since: 2020 (2020년부터 관계가 시작됨)
관계 생성 의미 : 홍길동은 김철수를 2020년부터 알게 됨
*/

// MATCH 구문으로 두 개의 기존 노드를 찾음
MATCH 
  (a:Person {name: "홍길동"}),  // Person 레이블을 가진 이름이 홍길동인 노드를 찾아 변수 a에 할당
  (b:Person {name: "김영수"})   // Person 레이블을 가진 이름이 김철수인 노드를 찾아 변수 b에 할당

// CREATE 구문으로 두 노드 사이에 방향성 있는 관계 생성
CREATE (a)-[r:KNOWS {since: 2020}]->(b)  // a에서 b로 향하는 KNOWS 타입의 관계 r 생성, since 속성 추가

// RETURN 구문으로 노드와 관계 정보 반환
RETURN a, b, r  // 생성된 관계와 연결된 두 노드 정보를 함께 반환
"""

graph.query(cypher_query)

In [ ]:
######################################################
#  기존에 생성된 노드들 사이에 관계를 추가하는 Cypher 쿼리
######################################################
cypher_query = """
/*
관계 레이블 : LIVES_IN (거주한다는 의미의 관계 타입)
관계 방향 : a -> c (이영희가 서울에 산다는 방향성)
관계 속성 : since: 2020 (2020년부터 관계가 시작됨)
관계 생성 의미 : 이영희는 서울에 2020년부터 거주하기 시작함
*/

// MATCH 구문으로 두 개의 기존 노드를 찾음
MATCH 
  (a:Person {name: "최영희"}),  // Person 레이블을 가진 이름이 이영희인 노드를 찾아 변수 a에 할당
  (c:City {name: "서울"})       // City 레이블을 가진 이름이 서울인 노드를 찾아 변수 c에 할당

// MATCH로 조회한 노드를 이용해 CREATE 구문으로 두 노드 사이에 방향성 있는 관계 생성한다.
CREATE (a)-[r:LIVES_IN {since: 2020}]->(c)  // a에서 c로 향하는 LIVES_IN 타입의 관계 r 생성, since 속성 추가

// RETURN 구문으로 노드와 관계 정보 반환
RETURN a, c, r  // 생성된 관계와 연결된 두 노드 정보를 함께 반환
"""
graph.query(cypher_query)

In [ ]:
########################################
#  노드와 관계를 한 번에 생성하는 Cypher 쿼리
########################################
cypher_query = """
// 노드 레이블 : Person, City (각각 사람과 도시를 나타내는 레이블)
// CREATE 구문으로 노드와 관계를 동시에 생성
CREATE 
  // a 변수에 Person 레이블을 가진 노드 생성, 속성으로 name="손흥민" 설정
  (a:Person {name: "손흥민"})-[r:PLAY_FOR]-> (t:Team {name:"LAFC"})
  // t 변수에 Team 레이블을 가진 노드 생성, 속성으로 name="LAFC" 설정
  // PLAY_FOR 관계는 손흥민과 LAFC의 관계 즉 손흥민이 LAFC에서 뛰고 있음을 나타낸다.

// RETURN 구문으로 생성된 두 노드 정보 반환
RETURN a, r, t
"""

graph.query(cypher_query)

In [ ]:
###########################################################
#  기존에 생성된 노드들 사이에 여러 관계를 추가하는 Cypher 쿼리
###########################################################
cypher_query = """
/*
첫 번째 관계 정보
관계 레이블 : LIVES_IN (거주한다는 의미의 관계 타입)
관계 방향 : a -> c (홍길동이 서울에 산다는 방향성)
관계 속성 : since: 2010 (2010년부터 관계가 시작됨)
관계 생성 의미 : 홍길동은 서울에 2010년부터 거주하기 시작함

두 번째 관계 정보
관계 레이블 : KNOWS (안다/알고 있다는 의미의 관계 타입)
관계 방향 : a -> b (홍길동이 손흥민을 안다는 방향성)
관계 속성 : since: 2020 (2020년부터 관계가 시작됨)
관계 생성 의미 : 홍길동은 손흥민을 2020년부터 알게 됨

세 번째 관계 정보
관계 레이블 : KNOWS (안다/알고 있다는 의미의 관계 타입)
관계 방향 : b -> a (손흥민이 홍길동을 안다는 방향성)
관계 속성 : since: 2020 (2020년부터 관계가 시작됨)
관계 생성 의미 : 손흥민은 홍길동을 2020년부터 알게 됨
두 번째와 세 번째 관계를 통해 양방향 관계(b <-> a)를 표현함
*/
// MATCH 구문으로 세 개의 기존 노드를 찾음

MATCH 
  (a:Person {name: "홍길동"}),  // Person 레이블을 가진 이름이 홍길동인 노드를 찾아 변수 a에 할당
  (b:Person {name: "손흥민"}),  // Person 레이블을 가진 이름이 박지성인 노드를 찾아 변수 b에 할당
  (c:City {name: "서울"})       // City 레이블을 가진 이름이 서울인 노드를 찾아 변수 c에 할당
  
// CREATE 구문으로 노드들 사이에 방향성 있는 관계 생성
CREATE (a)-[r1:LIVES_IN {since: 2010}]->(c)  // a에서 c로 향하는 LIVES_IN 타입의 관계 r1 생성, since 속성 추가
CREATE (a)-[r2:KNOWS {since: 2020}]->(b)     // a에서 b로 향하는 KNOWS 타입의 관계 r2 생성, since 속성 추가
CREATE (b)-[r3:KNOWS {since: 2020}]->(a)     // b에서 a로 향하는 KNOWS 타입의 관계 r3 생성, since 속성 추가
"""

graph.query(cypher_query)

In [ ]:
######################
# schema 조회
######################
graph.refresh_schema() # 변경된 부분을 반영 후 조회
print(graph.schema)

In [ ]:
graph.structured_schema

### **MATCH** 
- MATCH절은 그래프에서 특정 패턴(노드, 관계, 경로)을 **조회하는** 읽기(read) 명령어이다. SQL의 SELECT에 해당한다.
- `MATCH`만으로는 결과를 보여주지 않으며 항상 `RETURN`(또는 쓰기 절)과 함께 사용
- 노드/관계 자체에 변수를 붙여야 이후에 사용 가능
- 관계 방향은 `->`, `<-` 으로 지정 (`-` 양방향)
   
   ```cypher
   // 기본구문
   MATCH 패턴
   [WHERE 조건1 AND/OR 조건2 ...]
   RETURN ...
   [ORDER BY 정렬기준컬럼 ASC|DESC]

   // 모든 노드 찾기
   MATCH (n) RETURN n;

   // 특정 라벨의 노드 찾기
   MATCH (m:Movie) RETURN m;

   // 속성 조건이 있는 노드 찾기
   MATCH (m:Movie {title: '기생충'}) RETURN m;

   // 관계 패턴 찾기
   MATCH (p:Person)-[r:DIRECTED]->(m:Movie) RETURN p, r, m;

   // 관계 전체를 변수에 저장해서 반환
   MATCH path = (:Person)-[:DIRECTED]->(:Movie) RETURN path
   ```

#### RETURN
- `RETURN` 절은 쿼리의 결과를 어떻게 반환할지 결정한다. 단순한 노드 반환부터 표현식, 별칭, 중복 제거까지 가능하다.
- **기본 문법:**

   ```cypher
   RETURN 변수, 변수.속성, 표현식 [AS 별칭]
   RETURN DISTINCT ...   -- 중복 제거
   ```

- **반환 가능한 것들:**
  - 노드 자체 (`RETURN n`)
  - 속성 (`RETURN n.name`)
  - 관계 (`RETURN r`)
  - 계산 결과 (`RETURN n.born + 10`)
  - 문자열 결합 (`RETURN n.firstName + ' ' + n.lastName`)
  - 컬렉션 (`RETURN [n.name, n.born]`)

In [ ]:
###############################
#  모든 노드 조회
###############################
cypher_query = """  
// (n): 노드들을 변수 n에 할당
MATCH (n)
RETURN n
"""
graph.query(cypher_query)

In [ ]:
######################################
# 특정 Label의 노드 조회 및 특정 속성 출력
######################################
cypher_query = """
MATCH (n: Person) // Person Label의 노드들 조회해서 변수 n에 할당
RETURN n.name          // Person의 이름 속성만 출력
"""
graph.query(cypher_query)

In [ ]:
###############################################################
# 노드와 노드간의 관계를 조회
# (Person 레이블을 가진 노드와 City 레이블을 가진 노드 사이의 관계 조회)
###############################################################
cypher_query = """  

/*
(p:Person): Person 타입의 노드를 p 변수에 할당
-[r:LIVES_IN]->: LIVES_IN 타입의 관계를 r 변수에 할당하고, 방향은 Person에서 City로 향함
(c:City): City 타입의 노드를 c 변수에 할당
*/
MATCH (p:Person)-[r:LIVES_IN]->(c:City)

/*
RETURN: 쿼리 결과로 반환할 데이터를 지정
p.name AS Name: Person 노드의 name 속성을 'Name'이라는 별칭으로 반환
c.name AS City: City 노드의 name 속성을 'City'라는 별칭으로 반환
결과적으로 사람 이름과 그 사람이 사는 도시 이름을 테이블 형태로 반환
*/
RETURN p.name AS Name, c.name AS City
"""

graph.query(cypher_query)

#### **WHERE**: 조회 조건 지정
   - `WHERE` 절은 `MATCH` 결과에 추가 조건을 붙여 필터링한다. SQL의 `WHERE`와 거의 동일하다.
   - **기본 문법:**

      ```cypher
      MATCH 패턴
      WHERE 조건1 AND/OR 조건2 ...
      RETURN ...
      ORDER BY 정렬기준컬럼 ASC|DESC
      ```
- **비교/연산자:**

   | 연산자 | 의미 | 사용예시|
   |--------|------|---------|
   | `=`, `<>` | 같다 / 다르다 |WHERE n.age = 30, WHERE n.age <> 30|
   | `<`, `<=`, `>`, `>=` | 범위 비교 |WHERE n.age > 30, WHERE n.age <= 30|
   | `IN [...]` | 리스트 포함 | WHERE n.age IN [20, 30, 40]|
   | `STARTS WITH` | 시작하는 문자열 검색 |WHERE n.name STARTS WITH 'Kim'|
   | `ENDS WITH`| 끝나는 문자열 검색 |WHERE n.name ENDS WITH 'son'|
   | `CONTAINS` | 포함하는 문자열 검색 |n.name CONTAINS 'abc'|
   | `=~` | 정규 표현식식 매칭 |WHERE n.email =~ '.*@gmail.com'|
   | `IS NULL`, `IS NOT NULL` | NULL 체크 | WHERE n.age IS NULL, WHERE n.age IS NOT NULL|
   | `AND`, `OR`, `NOT` | 논리 연산 | WHERE n.age = 30 AND n..name ENDS WITH 'son'|

In [ ]:
###############################################################
# 조건 지정 (WHERE 구문)
# WHERE 구문은 MATCH로 찾은 패턴에 추가 조건을 적용할 때 사용
###############################################################
cypher_query = """
// Person 레이블을 가진 노드 중에서 age 속성이 20 이상인 사람 조회
MATCH (p:Person)
WHERE p.age >= 20

RETURN p
"""

graph.query(cypher_query)

In [ ]:
cypher_query = """
// Person 레이블을 가진 노드 중에서 age 속성이 20 이상이고 사는 도시가 서울인 노드 조회
// MATCH 구문으로 Person과 City 노드 간의 LIVES_IN 관계를 조회
MATCH (p:Person)-[r:LIVES_IN]->(c:City)

/*
WHERE 구문으로 조건을 지정:
1. p.age >= 20: Person 노드의 나이가 30 이상인 경우만 선택
2. c.name = "서울": City 노드의 이름이 "서울"인 경우만 선택
AND 연산자를 사용하여 두 조건을 모두 만족하는 노드만 필터링
*/
WHERE p.age >= 20 AND c.name = "서울"

// 조건을 만족하는 Person과 사는 City 노드를 반환
RETURN p, c

// 조건을 만족하는 사람의 이름과 도시명을 반환
// RETURN p.name AS 이름, c.name AS 사는도시
"""

graph.query(cypher_query)

#### ORDER BY / LIMIT / SKIP
- 쿼리 결과를 정렬하고, 일부만 가져오고, 건너뛴다.
- **기본 문법:**

    ```cypher
    RETURN ...
    ORDER BY 컬럼1 [ASC|DESC], 컬럼2 [ASC|DESC]
    SKIP N        -- 앞에서 N개 건너뛰기
    LIMIT M;      -- 최대 M개만 반환
    ```

- `ORDER BY`의 기본은 오름차순(`ASC`). 내림차순은 `DESC`.
- `SKIP`과 `LIMIT`은 페이징(pagination)에 자주 사용된다.

In [ ]:
#################################################
#  나이 순 정렬 - ORDER BY와 LIMIT 사용 예제
#################################################
cypher_query = """
// Person 레이블을 가진 모든 노드를 조회

MATCH (p:Person)

// 찾은 Person 노드 전체를 결과로 반환
RETURN p

/*
 ORDER BY 절을 사용하여 결과를 정렬
 p.age: Person 노드의 age 속성을 기준으로 정렬
 - 정렬방식
   - ASC: 오름차순 정렬 (작은 값에서 큰 값 순서로 정렬: Default)
   - DESC를 사용하면 내림차순 정렬.
*/
ORDER BY p.age ASC


// LIMIT 절을 사용하여 반환되는 결과의 수를 제한 한다. (LIMIT 개수)
LIMIT 2
"""


result = graph.query(cypher_query)

print("나이 순 정렬 결과:")

for record in result:
    print(record)

In [ ]:
#######################################
#  SKIP - 조회결과에서 앞 n개를 건너 뛴다.
#######################################
cypher_query = """
// Person 레이블을 가진 모든 노드를 검색.
MATCH (p:Person)

// 찾은 Person 노드 전체를 결과로 반환.
RETURN p

// ORDER BY 절을 사용하여 결과를 정렬한다. 
// p.age: Person 노드의 age 속성을 기준으로 정렬,  ASC: 오름차순 정렬 (작은 값에서 큰 값 순서로 정렬)
ORDER BY p.age ASC

// SKIP 절을 사용하여 결과의 처음 1개를 건너뛴다.
SKIP 1

// LIMIT 절을 사용하여 반환되는 결과의 수를 제한
// ORDER BY p.age ASC, SKIP 1, LIMIT 1 => 두 번째로 나이가 어린 사람만 조회
LIMIT 1
"""

result = graph.query(cypher_query)
print("나이 순 정렬 결과:")
for record in result:
    print(record)

### OPTIONAL MATCH
- `MATCH`는 관계 패턴이 일치하지 않으면 결과 행 자체가 조회되지 않는다. 반면 `OPTIONAL MATCH`는 일치하지 않아도 행을 유지하고 매칭되지 않은 변수는 `null`이 된다. SQL의 `LEFT OUTER JOIN`과 유사하다.
- **기본 문법:**

    ```cypher
    MATCH (a:Label)
    OPTIONAL MATCH (a)-[:REL]->(b)
    RETURN
    ```
- master 정보를 match로 찾고 부가 정보를 optional match로 정의한다.

In [ ]:
#############################
# OPTIONAL MATCH
#############################
cypher_query = """
// Person노드와 그가 Play하는 Team 노드들을 조회. 단 팀이 없어도 모든 Person 들은 다 나와야 한다.
MATCH (p: Person)
OPTIONAL MATCH (p) -[:PLAY_FOR]-> (t: Team)
return p, t
"""

result = graph.query(cypher_query)
for d in result:
    print(d)
    print("---------------------------"*3)

In [ ]:
cypher_query = """
// City노드와 거기에 사는 Person 노드들을 모두 조회. 단 사는 사람이 없어도 City는 모두 나와야 한다.
MATCH (c: City)
OPTIONAL MATCH (c) <-[:LIVES_IN]- (p: Person)
return c, p
"""

result = graph.query(cypher_query)
for d in result:
    print(d)
    print("---------------------------"*3)

### **MERGE**

- `MERGE`는 "있으면 찾고(MATCH), 없으면 만든다(CREATE)"
- `MERGE`는 **명시한 모든 속성**을 매칭 키로 사용한다. 그래서 매칭 키 외 속성은 `ON CREATE SET`에서 추가하는 것이 안전하다.
  - `MERGE (p:Person {name:'홍길동', age: 30})` 은 name이 홍길동이고 age가 30인 Person 이 있으면 조회 없으면 생성한다.
- **기본 문법:**

    ```cypher
    // 노드 MERGE
    MERGE (n:Label {key: value})

    // 관계 MERGE (보통 양 노드를 먼저 MATCH/MERGE한 뒤)
    MATCH (a), (b) WHERE ...
    MERGE (a)-[:REL]->(b)

    // MERGE에 따른 추가 동작
    // ON CREATE SET: 생성했을 때 실행되어 속성을 추가한다.
    // ON MERGE SET: 조회했을 때 실행되어 속성을 추가/변경한다.
    MERGE (n:Label {id: 1})
    ON CREATE SET n.createdAt = timestamp(), n.count = 1
    ON MATCH SET n.updatedAt = timestamp(), n.count = n.count + 1
    ```

In [ ]:
############################################################
# MERGE: 조회 또는 생성
# 이름이 손흥민인 사람이 있으면 조회하고 없으면 생성한다.
############################################################
cypher_query = """
// 이름이 손흥민인 사람이 있으면 조회하고 없으면 생성한다.
MERGE (p: Person {name:"손흥민"})

// 이름이 손흥민이고 나이가 30인 사람이 있으면 조회하고 없으면 생성한다.
//MERGE (p: Person {name:"손흥민", age: 30})
RETURN p
"""

result = graph.query(cypher_query)
for d in result:
    print(d)
    print("---------------------------"*3)

In [ ]:
##################################################################################
# 이름이 박지성인 사람이 있으면 조회하고 없으면 생성한다.
## 박지성이 생성되었으면 p.address = '수원' 속성을 추가한다.
## 박지성이 조회되었으면 p.address = '맨체스터' 속성을 추가 또는 변경한다.
##################################################################################
cypher_query = """
MERGE (p: Person {name:"박지성"})
// 박지성이 생성되었으면 p.address = '수원' 속성을 추가한다.
// 박지성이 조회되었으면 p.address = '맨체스터' 속성을 추가 또는 변경한다.
ON CREATE SET p.address = '수원'
ON MATCH SET p.address = '맨체스터'
RETURN p
"""

result = graph.query(cypher_query)
for d in result:
    print(d)
    print("---------------------------"*3)

In [ ]:
##########################################################################
# 관계가 있으면 조회하고 없으면 추가한다.
## 김영수와 서울 사이에 LIVES_IN 관계가 있으면 조회하고 없으면 생성한다.
##########################################################################
cypher_query = """

// 이름이 김영수인 사람과 도시 이름이 서울인 노드를 조회한다.
MATCH (p:Person{name:'김영수'}), (c:City {name: '서울'})

// 김영수와 서울 사이에 LIVES_IN 관계가 있으면 조회하고 없으면 생성한다.
MERGE (p) -[:LIVES_IN]-> (c)
RETURN p, c
"""

result = graph.query(cypher_query)
for d in result:
    print(d)
    print("---------------------------"*3)

### **SET** 
- SET` 절은 이미 존재하는 노드나 관계의 **속성(property), 라벨(label)을 추가하거나 수정**할 때 사용한다.
- CREATE가 새 노드나 관계를 만드는 절이라면, SET은 조회된 데이터의 값을 변경하는 절이다.

- **기본 문법:**

   ```cypher
   MATCH 패턴
   SET n.속성 = 값                // 단일 속성 변경/추가
   SET n.속성1 = v1, n.속성2 = v2 // 속성 변경/추가 (한개 속성=값)
   SET n += {key: value, ...}     // 속성 변경/추가 (map객체를 이용해서 한번에 여러 개 병합)
   SET n = {key: value, ...}      // 전체 교체(주의: 기존 속성 삭제됨)
   SET n:NewLabel                 // 라벨 추가
   ```

In [ ]:
#######################################
#  노드 속성 업데이트 (SET 구문 사용)
#######################################
cypher_query = """
// Person 레이블을 가진 노드 중에서 name 속성이 "홍길동"인 노드를 조회
// {name: "홍길동"} 형태로 속성 조건을 지정하여 특정 노드를 선택
MATCH (p:Person {name: "홍길동"})

/*
SET 구문을 사용하여 찾은 노드의 속성을 업데이트
p.age = 35: 'p'로 참조된 노드의 'age' 속성 값을 35로 설정
해당 속성이 이미 존재하면 값을 변경하고, 없으면 새로 생성
*/
SET p.age = 35, p.hobby = "게임"

RETURN p
"""

graph.query(cypher_query)

In [ ]:
cypher_query = """
// Person 레이블을 가진 노드 중에서 name 속성이 "손흥민"인 노드를 조회
// {name: "손흥민"} 형태로 속성 조건을 지정하여 특정 노드를 선택
MATCH (p:Person {name: "손흥민"})

/*
SET 구문을 사용하여 찾은 노드의 속성을 업데이트
age는 33, email은 'son@a.com' 으로 변경
해당 속성이 이미 존재하면 값을 변경하고, 없으면 새로 생성
*/
SET p += {age:33, email:'son@a.com'}

RETURN p
"""

graph.query(cypher_query)

In [ ]:
############################
#  레이블 추가 및 속성 설정 
############################
cypher_query = """
/*
레이블, 속성을 추가할 노드 조회
name 속성이 "홍길동"인 노드를 조회. (Label이 삭제된 노드)
MATCH (p {name: "홍길동"})

/*
SET 명령어를 사용하여 레이블추가, 속성 추가
  1. p:Person - 노드에 Person 레이블을 추가한다.
  2. p.age = 35 - age 속성을 35로 설정한다. (이전에 제거된 속성을 다시 추가)
SET 구문은 여러 작업을 쉼표로 구분하여 한 번에 수행할 수 있다.
*/

SET p:Person, p.age = 35

RETURN p
"""

graph.query(cypher_query)

### **DELETE**, **DETACH DELETE**
- `DELETE`: 노드/관계 삭제
  - Neo4j는 관계가 연결된 노드를 `DELETE`로 삭제할 수 없다.
- `DETACH DELETE`: 노드와 그에 연결된 모든 관계를 함께 삭제
- **기본 문법:**

   ```cypher
      // 관계만 삭제
      MATCH (a)-[r:REL]->(b) DELETE r;

      // 노드만 삭제 (연결 관계가 없을 때)
      MATCH (n:Label) DELETE n;

      // 노드 + 연결된 모든 관계 삭제
      MATCH (n:Label) DETACH DELETE n;

      // 전체 그래프 초기화
      MATCH (n) DETACH DELETE n;

   ```

In [ ]:
###################
#  노드 삭제
###################
cypher_query = """
// Person 레이블을 가진 노드 중에서 name 속성이 "최영희"인 노드를 찾는다
// {name: "최영희"} 형태로 속성 조건을 지정하여 특정 노드를 조회한다.
MATCH (p:Person {name: "최영희"})

/* 
DELETE 명령어를 사용하여 찾은 노드를 삭제한다.
주의: 이 방식은 관계가 없는 노드만 삭제 가능하다. 만약 노드에 관계가 있는 경우 ConstraintValidationFailed 오류가 발생
관계가 있는 노드를 삭제하려면 DETACH DELETE를 사용해야 한다.
*/
DELETE p
"""

graph.query(cypher_query)

In [ ]:
#################################################
#  관계가 있는 노드 삭제 (DETACH DELETE 사용)
#################################################
cypher_query = """
// Person 레이블을 가진 노드 중에서 name 속성이 "최영희"인 노드를 찾는다.
MATCH (p:Person {name: "최영희"})

// DETACH DELETE 명령어를 사용하여 노드와 그에 연결된 모든 관계를 함께 삭제한다.
// 이 명령어는 먼저 노드에 연결된 모든 관계를 제거한 후 노드 자체를 삭제한다.
DETACH DELETE p
"""

graph.query(cypher_query)

In [ ]:
################
# 관계를 삭제
################
cypher_query = """
/*
Person 레이블을 가진 노드 중에서 name 속성이 "홍길동"인 노드와 "김영수"인 노드 사이의 관계를 찾는다
(a:Person {name: "홍길동"}) - 시작 노드를 a 변수에 할당하고 name이 "홍길동"인 Person 노드로 지정한다.
-[r]-> - 방향성이 있는 관계를 r 변수에 할당한다. (관계 유형은 지정하지 않아 모든 유형의 관계가 대상이 된다.)
(b:Person {name: "김영수"}) - 도착 노드를 b 변수에 할당하고 name이 "김영수"인 Person 노드로 지정한다
*/

MATCH 
  (a:Person {name: "홍길동"})-[r]->(b:Person {name: "김영수"})

DELETE r
"""
graph.query(cypher_query)

### **REMOVE**
- 레이블이나 속성 제거
   ```cypher
   
      // 속성 삭제
      MATCH (n) REMOVE n.속성명;

      // 노드의 라벨 삭제
      MATCH (n) REMOVE n:라벨명;
   ```

In [ ]:
############################################ 
# 속성(Property) 제거 (REMOVE 구문 사용)
############################################  
cypher_query = """
// 삭제할 속성을 가진 노드를 먼저 찾는다. 
// Person 레이블을 가진 노드 중에서 name 속성이 "홍길동"인 노드를 조회
MATCH (p:Person {name: "홍길동"})

/*
REMOVE 명령어를 사용하여 찾은 노드의 age 속성을 제거
이 명령은 노드 자체나 다른 속성은 그대로 유지하고 지정된 속성만 삭제한다.
속성이 존재하지 않는 경우에도 오류 없이 실행된다.
*/
REMOVE p.age

// 변경된 노드를 결과로 반환하여 출력할 수있게 한다.
RETURN p
"""
graph.query(cypher_query)

In [ ]:
######################################
# 레이블(Label) 제거 (REMOVE 구문 사용)
######################################
cypher_query = """
// 레이블을 삭제할 노드를 먼저 조회한다.
// Person 레이블을 가진 노드 중에서 name 속성이 "홍길동"인 노드 찾아 p 변수에 할당한다.
MATCH (p:Person {name: "홍길동"})

/*
REMOVE 명령어를 사용하여 찾은 노드에서 Person 레이블을 제거한다.
이 명령은 노드 자체나 속성은 그대로 유지하고 지정된 레이블만 삭제한다.
레이블이 제거되면 해당 노드는 더 이상 Person으로 분류되지 않는다.
노드에 다른 레이블이 있다면 그 레이블은 유지된다.
*/
REMOVE p:Person

// 변경된 노드를 결과로 반환하여 출력할 수있게 한다.
RETURN p
"""

graph.query(cypher_query)